# Data Cleaning Step

The purpose of this notebook is to document the steps to clean up the dataset to prepare it for future analysis. The notebook will follow the following cleaning steps:

1. coerce passenger count and ratecodeID into integer data types
2. remove those rows with questionable pick up and drop off times
3. remove the rows where the drop off time is earlier than the pick up time
4. remove the rows where the pick up time is outside of Janurary
5. mark the rows where there are negative totals
6. mark the rows where there are negative totals and there are reversal positive matching rows

In [2]:
# Imports

import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# Read in the files

df = pd.read_parquet("../data/raw/yellow_tripdata_2026-01.parquet")
taxi_zone = pd.read_csv("../data/raw/taxi_zone_lookup.csv")

In [4]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [5]:
taxi_zone.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


### Step 1: Coerce Passenger Count into Integer Data Types

In [6]:
df.dtypes.reset_index()

,index,0
0,VendorID,int32
1,tpep_pickup_datetime,datetime64[us]
2,tpep_dropoff_datetime,datetime64[us]
3,passenger_count,float64
4,trip_distance,float64
5,RatecodeID,float64
6,store_and_fwd_flag,str
7,PULocationID,int32
8,DOLocationID,int32
9,payment_type,int64


In [7]:
df['RatecodeID'] = df['RatecodeID'].astype('Int32')
df[['RatecodeID']].sample(10)

,RatecodeID
3348309,<NA>
3219449,<NA>
1196056,1
2686217,<NA>
527313,1
3470091,<NA>
3329858,<NA>
2569786,1
2455448,1
2524621,1


In [8]:
df['passenger_count'] = df['passenger_count'].astype('Int32')
df[['passenger_count']].sample(10)

,passenger_count
1582672,1
2686324,<NA>
2270262,1
2016472,2
2700910,<NA>
3104715,<NA>
735523,1
1991334,1
3205261,<NA>
925816,1


### Step 2: Remove columns with questionable pick up times (dropff time that is earlier thatn pick up time)

In [9]:
df[df['tpep_pickup_datetime'] > df['tpep_dropoff_datetime']]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1465434,1,2026-01-17 19:30:00,2026-01-17 19:18:18,1,16.5,99,N,46,86,1,66.5,0.0,0.5,0.0,0.0,0.0,67.0,0.0,0.0,0.0


In [10]:
df.drop(index=1465434, inplace=True)

In [11]:
df[df['tpep_pickup_datetime'] > df['tpep_dropoff_datetime']]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee


this row has been removed

## Step 3: Remove the rows where the pick up time is earlier than Jaurary

In [12]:
start_date = pd.Timestamp("2026-01-01")
outside_bound_list = df[df['tpep_pickup_datetime'] < start_date].index.tolist()
outside_bound_list

[980, 981, 982, 1404, 2289, 5181]

In [13]:
df.drop(index=outside_bound_list, inplace=True)

In [14]:
df[df['tpep_dropoff_datetime'] < start_date]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee


In [15]:
end_date = pd.Timestamp('2026-02-01')
outside_bound_list = df[df['tpep_pickup_datetime'] >= end_date].index.tolist()
outside_bound_list

[2635466]

In [16]:
df[df['tpep_pickup_datetime'] >= end_date]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
2635466,2,2026-02-01 00:45:01,2026-02-01 01:03:08,1,1.78,1,N,79,249,1,15.6,1.0,0.5,5.34,0.0,1.0,26.69,2.5,0.0,0.75


In [17]:
df.drop(index=outside_bound_list, inplace=True)

In [18]:
df[df['tpep_pickup_datetime'] >= end_date]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee


pick up times that are later than end date are also dropped

### Step 4: Mark the rows where the totals are negative and also if there are reversal transactions

In [19]:
df['is_negative_total'] = df['total_amount'] < 0
df[['is_negative_total']].sample(10)

,is_negative_total
1330555,False
1702805,False
1741191,False
1431007,False
661448,False
1076349,False
1818910,False
472528,False
3040587,False
353457,False


In [20]:
df['is_matched_reversal'] = False

In [21]:
match_cols = [
    "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance"
]

negative_trips = df[df['is_negative_total'] == True]
positive_trips = df[df['is_negative_total'] == False]

matched_pairs = negative_trips.merge(positive_trips, on=match_cols, suffixes=("_neg", "_pos"))
matched_pairs.shape

(35712, 39)

In [22]:
matched_pairs.sample(10)

,VendorID_neg,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count_neg,trip_distance,RatecodeID_neg,store_and_fwd_flag_neg,PULocationID,DOLocationID,payment_type_neg,...,mta_tax_pos,tip_amount_pos,tolls_amount_pos,improvement_surcharge_pos,total_amount_pos,congestion_surcharge_pos,Airport_fee_pos,cbd_congestion_fee_pos,is_negative_total_pos,is_matched_reversal_pos
23742,2,2026-01-21 01:54:23,2026-01-21 02:03:01,1,1.85,1,N,229,68,4,...,0.5,0.0,0.00,1.0,17.15,2.5,0.00,0.75,False,False
25007,2,2026-01-22 06:53:25,2026-01-22 07:39:41,1,11.98,1,N,132,61,2,...,0.5,0.0,0.00,1.0,59.45,0.0,1.75,0.00,False,False
33031,2,2026-01-29 21:18:35,2026-01-29 21:33:36,1,2.95,1,N,148,229,4,...,0.5,0.0,0.00,1.0,22.75,2.5,0.00,0.75,False,False
9794,2,2026-01-09 10:00:50,2026-01-09 10:04:32,1,0.84,1,N,151,239,4,...,0.5,0.0,0.00,1.0,10.50,2.5,0.00,0.00,False,False
30685,2,2026-01-27 22:22:55,2026-01-27 22:47:46,1,11.67,1,N,132,96,4,...,0.5,0.0,0.00,1.0,49.25,0.0,1.75,0.00,False,False
20752,2,2026-01-18 11:52:14,2026-01-18 11:59:01,1,1.50,1,N,142,48,4,...,0.5,0.0,0.00,1.0,14.05,2.5,0.00,0.75,False,False
33140,2,2026-01-29 22:12:48,2026-01-29 22:39:24,1,13.95,1,N,132,226,2,...,0.5,0.0,0.00,1.0,56.95,0.0,1.75,0.00,False,False
27333,2,2026-01-23 22:27:18,2026-01-23 22:37:18,1,1.81,1,N,229,164,2,...,0.5,0.0,0.00,1.0,17.15,2.5,0.00,0.75,False,False
35485,2,2026-01-31 20:11:16,2026-01-31 20:32:12,1,1.52,1,N,246,230,4,...,0.5,0.0,0.00,1.0,24.15,2.5,0.00,0.75,False,False
18227,2,2026-01-16 12:25:15,2026-01-16 12:52:39,1,9.58,1,N,138,243,4,...,0.5,0.0,7.46,1.0,56.51,0.0,1.75,0.00,False,False


In [23]:
df["reversal_key"] = list(
    zip(
        df["tpep_pickup_datetime"],
        df["tpep_dropoff_datetime"],
        df["PULocationID"],
        df["DOLocationID"],
        df["trip_distance"],
        df["total_amount"]
    )
)

matched_negative_keys = set(
    zip(
        matched_pairs["tpep_pickup_datetime"],
        matched_pairs["tpep_dropoff_datetime"],
        matched_pairs["PULocationID"],
        matched_pairs["DOLocationID"],
        matched_pairs["trip_distance"],
        matched_pairs["total_amount_neg"]
    )
)

df['is_matched_reversal'] = df['reversal_key'].isin(matched_negative_keys)

In [24]:
df.sample(10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_negative_total,is_matched_reversal,reversal_key
2465889,1,2026-01-30 09:09:01,2026-01-30 09:24:29,1,0.90,1,N,140,140,1,...,1.00,0.0,1.0,19.25,2.5,0.00,0.75,False,False,"(2026-01-30 09:09:01, 2026-01-30 09:24:29, 140..."
363251,2,2026-01-05 22:39:25,2026-01-05 23:15:37,1,15.26,1,N,132,7,1,...,9.00,0.0,1.0,75.75,0.0,1.75,0.00,False,False,"(2026-01-05 22:39:25, 2026-01-05 23:15:37, 132..."
2613454,2,2026-01-31 19:31:41,2026-01-31 19:56:43,3,3.23,1,N,238,162,1,...,5.75,0.0,1.0,34.50,2.5,0.00,0.75,False,False,"(2026-01-31 19:31:41, 2026-01-31 19:56:43, 238..."
1374760,1,2026-01-16 19:36:30,2026-01-16 19:51:50,1,2.90,1,N,151,237,1,...,4.42,0.0,1.0,26.52,2.5,0.00,0.00,False,False,"(2026-01-16 19:36:30, 2026-01-16 19:51:50, 151..."
935276,2,2026-01-12 11:20:21,2026-01-12 11:29:42,1,1.09,1,N,68,158,1,...,2.95,0.0,1.0,17.70,2.5,0.00,0.75,False,False,"(2026-01-12 11:20:21, 2026-01-12 11:29:42, 68,..."
1753909,2,2026-01-21 11:08:42,2026-01-21 11:35:24,1,2.16,1,N,163,262,1,...,5.47,0.0,1.0,32.82,2.5,0.00,0.75,False,False,"(2026-01-21 11:08:42, 2026-01-21 11:35:24, 163..."
2109030,1,2026-01-24 20:09:06,2026-01-24 20:25:42,1,2.60,1,N,24,262,1,...,5.15,0.0,1.0,25.75,2.5,0.00,0.00,False,False,"(2026-01-24 20:09:06, 2026-01-24 20:25:42, 24,..."
1134523,2,2026-01-14 14:34:34,2026-01-14 14:40:30,1,0.74,1,N,170,234,1,...,1.00,0.0,1.0,13.65,2.5,0.00,0.75,False,False,"(2026-01-14 14:34:34, 2026-01-14 14:40:30, 170..."
1193116,2,2026-01-14 23:30:33,2026-01-14 23:47:52,1,3.52,1,N,161,231,1,...,1.50,0.0,1.0,25.65,2.5,0.00,0.75,False,False,"(2026-01-14 23:30:33, 2026-01-14 23:47:52, 161..."
276245,1,2026-01-04 19:06:11,2026-01-04 19:17:55,2,2.20,1,N,186,4,2,...,0.00,0.0,1.0,18.25,2.5,0.00,0.75,False,False,"(2026-01-04 19:06:11, 2026-01-04 19:17:55, 186..."


In [25]:
df.drop(columns=['reversal_key'], inplace=True)

In [26]:
df.sample(10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,is_negative_total,is_matched_reversal
2806230,2,2026-01-06 23:49:57,2026-01-07 00:05:01,<NA>,3.02,<NA>,NaN,90,232,0,...,0.5,0.00,0.00,1.0,26.18,NaN,NaN,0.75,False,False
790253,2,2026-01-10 17:41:32,2026-01-10 18:01:16,1,8.00,1,N,70,262,1,...,0.5,9.91,7.46,1.0,61.22,2.5,1.75,0.00,False,False
1290573,2,2026-01-15 21:03:26,2026-01-15 21:24:18,1,2.75,1,N,148,164,1,...,0.5,4.97,0.00,1.0,29.82,2.5,0.00,0.75,False,False
1105730,2,2026-01-14 09:18:48,2026-01-14 09:31:59,1,1.48,1,N,162,142,1,...,0.5,4.56,0.00,1.0,22.81,2.5,0.00,0.75,False,False
190586,2,2026-01-03 17:56:34,2026-01-03 18:07:42,1,1.00,1,N,79,148,1,...,0.5,2.00,0.00,1.0,17.45,2.5,0.00,0.75,False,False
1663267,1,2026-01-20 12:09:49,2026-01-20 12:52:28,1,8.60,99,N,181,48,1,...,0.5,0.00,0.00,0.0,36.00,0.0,0.00,0.00,False,False
2792592,2,2026-01-06 11:18:37,2026-01-06 11:45:47,<NA>,11.39,<NA>,NaN,234,243,0,...,0.5,0.00,0.00,1.0,50.02,NaN,NaN,0.75,False,False
3514445,2,2026-01-27 20:51:59,2026-01-27 21:11:31,<NA>,2.77,<NA>,NaN,164,231,0,...,0.5,0.00,0.00,1.0,60.21,NaN,NaN,0.75,False,False
2185637,2,2026-01-27 03:35:11,2026-01-27 04:03:29,1,10.13,1,N,132,177,2,...,0.5,0.00,0.00,1.0,46.45,0.0,1.75,0.00,False,False
3076947,2,2026-01-15 21:51:03,2026-01-15 22:02:28,<NA>,1.45,<NA>,NaN,79,211,0,...,0.5,0.00,0.00,1.0,18.31,NaN,NaN,0.75,False,False


In [27]:
df.columns.tolist()

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee',
 'is_negative_total',
 'is_matched_reversal']

In [29]:
df.to_csv("../data/raw/yellow_tripdata_2026-01_cleaned.csv", index=False)